# IV_03 — SQL y Supabase

## 1. Objetivo

Consultar datos de planta con SQL usando Supabase (PostgreSQL en la nube) o SQLite local como fallback.

## 2. Concepto — SQL esencial

| Comando | Uso en planta |
|---------|---------------|
| `SELECT` | Leer columnas |
| `WHERE` | Filtrar por equipo, fecha, calidad |
| `GROUP BY` | Agregar por turno, equipo |
| `JOIN` | Unir equipos con eventos o lecturas |

In [ ]:
import os
import sqlite3
from pathlib import Path

import pandas as pd

MOD_DIR = Path.cwd()
os.chdir(MOD_DIR)
DATA_DIR = MOD_DIR / "data"
OUTPUT_DIR = MOD_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / "ecosistema_local.db"

def init_local_db():
    """Carga CSV en SQLite local (fallback sin Supabase)."""
    if DB_PATH.exists():
        return
    conn = sqlite3.connect(DB_PATH)
    pd.read_csv(DATA_DIR / "equipos.csv").to_sql("equipos", conn, if_exists="replace", index=False)
    pd.read_csv(DATA_DIR / "lecturas_pi_export.csv", parse_dates=["timestamp"]).to_sql(
        "lecturas_pi", conn, if_exists="replace", index=False
    )
    pd.read_csv(DATA_DIR / "eventos_mantenimiento.csv", parse_dates=["inicio", "fin"]).to_sql(
        "eventos_mantenimiento", conn, if_exists="replace", index=False
    )
    conn.close()

def get_supabase_client():
    from dotenv import load_dotenv
    load_dotenv(MOD_DIR / ".env")
    url, key = os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY")
    if url and key:
        from supabase import create_client
        return create_client(url, key)
    return None

def query_sql(sql, params=()):
    init_local_db()
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return df


## 3. Consultas SQL locales (SQLite)

In [ ]:
# Equipos de la planta
query_sql("SELECT * FROM equipos")


In [ ]:
# Lecturas GOOD de vibración
query_sql("""
    SELECT timestamp, tag, valor, unidad
    FROM lecturas_pi
    WHERE quality = 'GOOD' AND tag LIKE '%VIBRATION%'
    ORDER BY timestamp
    LIMIT 10
""")


In [ ]:
# Equipos con más eventos de mantenimiento (JOIN)
query_sql("""
    SELECT e.codigo, e.area, COUNT(ev.equipo_id) AS num_eventos, AVG(ev.mttr_horas) AS mttr_prom
    FROM equipos e
    LEFT JOIN eventos_mantenimiento ev ON e.id = ev.equipo_id
    GROUP BY e.codigo, e.area
    ORDER BY num_eventos DESC
""")


## 4. Consultas con Supabase (si .env configurado)

In [ ]:
client = get_supabase_client()
if client:
    res = client.table("equipos").select("codigo, area, tipo").execute()
    df_supa = pd.DataFrame(res.data)
    print("Datos desde Supabase:")
    display(df_supa)
else:
    print("Sin .env — usando SQLite local. Configura Supabase para producción.")


## 5. Ejercicio práctico

Escribe una consulta SQL que devuelva el promedio de `valor` por `tag` solo para registros con `quality = 'GOOD'`.

In [ ]:
# Solución
query_sql("""
    SELECT tag, AVG(valor) AS promedio, COUNT(*) AS n
    FROM lecturas_pi
    WHERE quality = 'GOOD'
    GROUP BY tag
    ORDER BY promedio DESC
""")


## 6. Resumen y siguiente paso

- SQL es el lenguaje universal de bases de datos.
- Supabase ofrece PostgreSQL gestionado con API REST.
- SQLite local permite practicar sin conexión.

**Siguiente:** `IV_04_pipeline_python_datos.ipynb`